<a href="https://colab.research.google.com/github/Prajjwal2123/internship-assignments/blob/main/week_8_Prajjwal_Sharma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports & Setup**

In [1]:
import re
import json
import logging
from datetime import datetime

# Basic logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("SmartAgent")

# **Tools**

In [2]:
def calculator_tool(query: str) -> dict:
    """
    Extracts a math expression from the query and evaluates it safely.
    """
    try:
        # Extract numbers and basic math operators only
        expression = re.sub(r'[^0-9+\-*/().\s]', '', query)
        expression = expression.strip()

        if not expression:
            raise ValueError("No valid mathematical expression found")

        # Safe eval using a restricted namespace
        result = eval(expression, {"__builtins__": {}}, {})
        return {"success": True, "value": result, "expression_used": expression}

    except ZeroDivisionError:
        return {"success": False, "error": "Division by zero"}
    except Exception as e:
        return {"success": False, "error": f"Invalid expression: {str(e)}"}


def keyword_tool(query: str) -> dict:
    """
    Extracts simple keywords by removing stopwords and short tokens.
    """
    try:
        stopwords = {
            "the", "is", "in", "at", "of", "a", "an", "and", "or", "to",
            "for", "on", "with", "keywords", "extract", "find", "please",
            "me", "get", "from", "this", "that"
        }
        words = re.findall(r'\b[a-zA-Z]+\b', query.lower())
        keywords = [w for w in words if w not in stopwords and len(w) > 2]

        # Remove duplicates while preserving order
        seen = set()
        unique_keywords = []
        for w in keywords:
            if w not in seen:
                seen.add(w)
                unique_keywords.append(w)

        return {"success": True, "keywords": unique_keywords}

    except Exception as e:
        return {"success": False, "error": str(e)}


def general_response_tool(query: str) -> dict:
    """
    Handles general queries with a simple canned response.
    (You can later replace this with an LLM API call.)
    """
    try:
        response = f"I received your query: '{query}'. This is a general response since no specific tool matched."
        return {"success": True, "response": response}
    except Exception as e:
        return {"success": False, "error": str(e)}

# **Agent Routing Logic**

In [3]:
class SmartAgent:
    def __init__(self):
        self.name = "SingleAgentPipeline"

    def route_query(self, query: str) -> str:
        """
        Determines which tool to use based on keywords in the query.
        """
        query_lower = query.lower()

        if "calculate" in query_lower:
            return "calculation"
        elif "keywords" in query_lower:
            return "keywords"
        else:
            return "general"

    def run(self, query: str) -> dict:
        """
        Main entry point: routes the query, calls the right tool,
        and returns a structured JSON-style response.
        """
        logger.info(f"Received query: {query}")

        if not query or not isinstance(query, str) or not query.strip():
            logger.error("Empty or invalid query received")
            return {"type": "error", "result": "Query is empty or invalid"}

        try:
            intent = self.route_query(query)
            logger.info(f"Routed to intent: {intent}")

            if intent == "calculation":
                tool_result = calculator_tool(query)
                if tool_result["success"]:
                    return {"type": "calculation", "result": tool_result["value"]}
                else:
                    return {"type": "error", "result": tool_result["error"]}

            elif intent == "keywords":
                tool_result = keyword_tool(query)
                if tool_result["success"]:
                    return {"type": "keywords", "result": tool_result["keywords"]}
                else:
                    return {"type": "error", "result": tool_result["error"]}

            else:  # general
                tool_result = general_response_tool(query)
                if tool_result["success"]:
                    return {"type": "general", "result": tool_result["response"]}
                else:
                    return {"type": "error", "result": tool_result["error"]}

        except Exception as e:
            logger.exception("Unexpected error while processing query")
            return {"type": "error", "result": f"Unexpected error: {str(e)}"}

# **Running & Test**

In [4]:
agent = SmartAgent()

test_queries = [
    "calculate 12 * (5 + 3)",
    "extract keywords from this sentence about machine learning and pipelines",
    "tell me a fun fact",
    "calculate 10 / 0",
    ""
]

for q in test_queries:
    output = agent.run(q)
    print(f"Query: {q!r}")
    print(json.dumps(output, indent=2))
    print("-" * 50)

ERROR:SmartAgent:Empty or invalid query received


Query: 'calculate 12 * (5 + 3)'
{
  "type": "calculation",
  "result": 96
}
--------------------------------------------------
Query: 'extract keywords from this sentence about machine learning and pipelines'
{
  "type": "keywords",
  "result": [
    "sentence",
    "about",
    "machine",
    "learning",
    "pipelines"
  ]
}
--------------------------------------------------
Query: 'tell me a fun fact'
{
  "type": "general",
  "result": "I received your query: 'tell me a fun fact'. This is a general response since no specific tool matched."
}
--------------------------------------------------
Query: 'calculate 10 / 0'
{
  "type": "error",
  "result": "Division by zero"
}
--------------------------------------------------
Query: ''
{
  "type": "error",
  "result": "Query is empty or invalid"
}
--------------------------------------------------
